# Ye et al. (2014) Dataset
Consist of only fix projects: Eclipse_Platform_UI, AspectJ, Birt, SWT, JDT, Tomcat

In [1]:
import pandas as pd
import glob
import os

In [3]:
# Root directory of the Ye et al. dataset
ye_et_al_directory = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/Ye_et_al_Dataset'

In [5]:
'''
Loads all .xlsx files from a directory, combines them, and analyzes the bug counts for each project.
'''
print("Analyzing Ye et al. Dataset.....")

try:
    # Find all .xlsx files in the specified directory
    excel_files = glob.glob(os.path.join(ye_et_al_directory, '*.xlsx'))

    if not excel_files:
        print(f"Error: No .xlsx files found in '{ye_et_al_directory}'. Please check the path.")

    all_project_dfs = []

    # Loop through each file, read it, and add a project_name column
    for file_path in excel_files:
        # Extract project name from the filename (e.g. 'swt.xlsx' -> 'swt')
        project_name = os.path.basename(file_path).split('.')[0]

        df = pd.read_excel(file_path)
        df['project_name'] = project_name
        all_project_dfs.append(df)
        print(f"Loaded {len(df)} records from {project_name}.xlsx")

    # Combine all data into a single DataFrame
    combined_df = pd.concat(all_project_dfs, ignore_index=True)
    print(f"Successfully loaded a total of {len(combined_df)} records from {len(excel_files)} projects. ")

    # Analysis Step
    # Create full description column for duplicate checking
    # Use .astype(str) to handle potential non-string/missing values
    combined_df['bug_description'] = combined_df['summary'].astype(str) + combined_df['description'].astype(str)

    # Count total bug reports per project
    bug_counts = combined_df.groupby('project_name')['bug_id'].count().reset_index(name='Total_Bug_Reports')

    # Count duplicate bug reports per project
    duplicate_counts = combined_df.groupby('project_name').apply(
        lambda x: x.shape[0] - x['bug_description'].nunique()
    ).reset_index(name='Duplicate_Bug_Reports')

    # Combine and sort results
    results_df = pd.merge(bug_counts, duplicate_counts, on='project_name')
    sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False).reset_index(drop=True)

    print(" Ye et al Analysis Results")
    print(sorted_results.to_string())

except Exception as e:
    print(f"An error occurred: {e}")

Analyzing Ye et al. Dataset.....
Loaded 4151 records from swt.xlsx
Loaded 1056 records from tomcat.xlsx
Loaded 4178 records from birt.xlsx
Loaded 6495 records from eclipse_platform_ui.xlsx
Loaded 593 records from aspectj.xlsx
Loaded 6274 records from jdt.xlsx
Successfully loaded a total of 22747 records from 6 projects. 
 Ye et al Analysis Results
          project_name  Total_Bug_Reports  Duplicate_Bug_Reports
0  eclipse_platform_ui               6495                      0
1                  jdt               6274                      0
2                 birt               4178                      0
3                  swt               4151                      0
4               tomcat               1056                      0
5              aspectj                593                      0


/tmp/ipykernel_123853/1445323273.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = combined_df.groupby('project_name').apply(


## Extracting total bug reports, duplicate bug reports and total unique reports

In [4]:
'''
Loads all .xlsx files from a directory, combines them, and analyzes the bug counts for each project.
'''
print("Analyzing Ye et al. Dataset.....")

try:
    # Find all .xlsx files in the specified directory
    excel_files = glob.glob(os.path.join(ye_et_al_directory, '*.xlsx'))

    if not excel_files:
        print(f"Error: No .xlsx files found in '{ye_et_al_directory}'. Please check the path.")

    all_project_dfs = []

    # Loop through each file, read it, and add a project_name column
    for file_path in excel_files:
        # Extract project name from the filename (e.g. 'swt.xlsx' -> 'swt')
        project_name = os.path.basename(file_path).split('.')[0]
        df = pd.read_excel(file_path)
        df['project_name'] = project_name
        all_project_dfs.append(df)
        print(f"Loaded {len(df)} records from {project_name}.xlsx")

    # Combine all data into a single DataFrame
    combined_df = pd.concat(all_project_dfs, ignore_index=True)
    print(f"Successfully loaded a total of {len(combined_df)} records from {len(excel_files)} projects. ")

    # Analysis Step
    # Group by project and perform all aggregations using the 'commit' hash for accuracy
    repo_analysis = combined_df.groupby('project_name').agg(
        total_bug_reports=('bug_id', 'count'),
        unique_fix_count=('commit', 'nunique')
    ).reset_index() 

    # Calculate duplicate and unique bug report columns
    repo_analysis['duplicate_bug_reports'] = repo_analysis['total_bug_reports'] - repo_analysis['unique_fix_count']
    repo_analysis['total_unique_reports'] = repo_analysis['unique_fix_count']

    # Rename 'project_name' to 'repo' for consistency
    repo_analysis.rename(columns={'project_name': 'repo'}, inplace=True)

     # Add a placeholder for repo_link, as owner information is not in the data
    # This will need to be curated manually later if needed.
    repo_analysis['repo_link'] = 'Needs Manual Curation'

    # Select and reorder the final columns to match your request
    final_columns = [
        'repo', 'repo_link', 'total_bug_reports', 'duplicate_bug_reports', 'total_unique_reports'
    ]
    
    sorted_results = repo_analysis[final_columns].sort_values(
        by='total_bug_reports', ascending=False
    ).reset_index(drop=True)

    print(" Ye et al Analysis Results")
    print(sorted_results.to_string())

except Exception as e:
    print(f"An error occurred: {e}")

Analyzing Ye et al. Dataset.....
Loaded 4151 records from swt.xlsx
Loaded 1056 records from tomcat.xlsx
Loaded 4178 records from birt.xlsx
Loaded 6495 records from eclipse_platform_ui.xlsx
Loaded 593 records from aspectj.xlsx
Loaded 6274 records from jdt.xlsx
Successfully loaded a total of 22747 records from 6 projects. 
 Ye et al Analysis Results
                  repo              repo_link  total_bug_reports  duplicate_bug_reports  total_unique_reports
0  eclipse_platform_ui  Needs Manual Curation               6495                      0                  6495
1                  jdt  Needs Manual Curation               6274                      0                  6274
2                 birt  Needs Manual Curation               4178                      0                  4178
3                  swt  Needs Manual Curation               4151                      0                  4151
4               tomcat  Needs Manual Curation               1056                      0             

## Extracting unique extensions of the files from ground truth files

In [2]:
def get_file_extension(filepath):
    # Extracts the file extension from a given path.
    if '.' in os.path.basename(filepath):
        return os.path.splitext(filepath)[1]
    return 'no_extension'

In [5]:
def analyze_ye_et_al_unique_extensions(combined_df):
    '''
    Finds a single unique list of all groundt truth file extensions from the combined Ye et al.
    dataset DataFrame

    Args:
        combined_df (pd.DataFrame): The DataFrame containing all Ye et al. data
    '''
    print(" Finding unique ground truth file extensions for ye et al. dataset")

    if 'files' not in combined_df.columns:
        print("Error: 'files' column not found in the DataFrame.")
        return
    
    # making a copy to avoid modifying the original dataframe
    df = combined_df.copy()

    # Drop rows where the 'files' column is empty/NaN
    df.dropna(subset=['files'], inplace=True)

    # Split the space-separated string of files into a list of files
    df['files_list'] = df['files'].str.split('\n')

    # Explode the DataFrame to create on row per file
    all_files_df = df.explode('files_list')

    # Get the unique extension for each file using a set for automatic uniqueness
    unique_extensions = {get_file_extension(f) for f in all_files_df['files_list'] if f}

    print("Unique file extensions across all projects")
    print(sorted(list(unique_extensions)))


In [6]:
analyze_ye_et_al_unique_extensions(combined_df)

 Finding unique ground truth file extensions for ye et al. dataset
Unique file extensions across all projects
['.eclip', '.java', 'no_extension']


# MetaData Extraction
Set of methods to calculate all ten metadata for selected projects from Ye et al. dataset

1. LOC
2. age_years
3. median_bug_year
4. num_authors
5. num_commits
6. num_dependencies
7. polyglot_index
8. bug_density
9. code_complexity
10. bug_report_verbosity

In [23]:
# Import libraries
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import glob
import tempfile 

## Configuration 

In [22]:
def process_ye_et_al_dataset(data_directory, clone_dir):
    '''
    Loads, combines, and processes the Ye et al. dataset to prepare it for metadata extraction.
    This method will:
    a) Load data from all .xlsx files.
    b) Map folder names to correct 'owner/repo' names
    c) Add a 'language' column.
    d) Rename the 'commit' column to 'after_fix_sha'
    e) Find the parent commit for each fix to create the 'before_fix_sha'

    Args:
        data_directory (str): The path to the folder containing the .xlsx files.
        clone_dir (str): The path to the parent directory where repos are cloned.

    Returns:
        pd.DataFrame: A fully processed DataFrame ready for analysis/
    '''
    print("Processing Ye et al. Dataset")

    # The manual mapping from filename to correct repo name
    repo_name_map = {
        'tomcat': 'apache/tomcat',
        'aspectj': 'eclipse-aspectj/aspectj',
        'birt': 'eclipse-birt/birt',
        'jdt': 'eclipse-jdt/eclipse.jdt.ui',
        'eclipse_platform_ui': 'eclipse-platform/eclipse.platform.ui',
        'swt': 'eclipse-platform/eclipse.platform.swt'
    }

    # 1. Load and combine data from all .xlsx files
    excel_files = glob.glob(os.path.join(data_directory, '*.xlsx'))
    if not excel_files:
        print(f"Error: No .xlsx files found in '{data_directory}'.")
        return None
    
    all_project_dfs = []
    for file_path in excel_files:
        project_key = os.path.basename(file_path).split('.')[0]
        df = pd.read_excel(file_path)
        df['project_key'] = project_key # Temporary key for mapping
        all_project_dfs.append(df)

    ye_et_al_df = pd.concat(all_project_dfs, ignore_index=True)
    print(f"Successfully loaded {len(ye_et_al_df)} total bug reports.")

    # 2. Add and rename columns
    ye_et_al_df['language'] = 'java'
    ye_et_al_df['repo_name'] = ye_et_al_df['project_key'].map(repo_name_map)
    ye_et_al_df.rename(columns={'commit': 'after_fix_sha'}, inplace=True)
    ye_et_al_df['bug_id'] = ye_et_al_df['project_key'].astype(str) + '-' + ye_et_al_df['bug_id'].astype(str)
    ye_et_al_df.drop(columns=['project_key'], inplace=True) # Clean up temp key

    # 3. Find the 'before_fix_sha' for each bug report
    ye_et_al_df['before_fix_sha'] = None
    grouped = ye_et_al_df.groupby('repo_name')

    for repo_name, group in tqdm(grouped, desc='Finding Parent Commits'):
        repo_path = os.path.join(clone_dir, 'java', repo_name.replace('/', '_'))
        
        if not os.path.exists(repo_path):
            tqdm.write(f"Warning: Cloned repo not found for {repo_name}. Skipping...")
            continue

        try:
            repo = git.Repo(repo_path)
            for index, row in group.iterrows():
                fix_commit_sha = row['after_fix_sha']
                try:
                    # Get the commit object for the fix
                    commit = repo.commit(fix_commit_sha)
                    # The first parent is the before_fix_sha
                    if commit.parents:
                        parent_sha = commit.parents[0].hexsha
                        ye_et_al_df.loc[index, 'before_fix_sha'] = parent_sha
                except git.exc.GitCommandError:
                    # This happens if the commit hash doesn't exist in the repo
                    tqdm.write(f"Warning: Commit {fix_commit_sha} not found in {repo_name}.")
                    continue
        
        except Exception as e:
            tqdm.write(f"An error occurred while processing {repo_name}: {e}")

    # Remove rows where 'before_fix_sha' could not be found
    initial_count = len(ye_et_al_df)
    ye_et_al_df.dropna(subset=['before_fix_sha'], inplace=True)
    final_count = len(ye_et_al_df)
    print(f"\n Removed {initial_count - final_count} rows with missing 'before_fix_sha' values")

    print("Processing complete....")
    return ye_et_al_df

In [24]:
# 1. Path to the input CSV file
# Must have columns: 'repo_name', 'language', 'total_unique_bug_report'
INPUT_CSV_PATH = "/home/cs21d002_eashaan/PhD/Objective1/benchmark_dataset_analysis/metadata/ye_metadata_input.csv"

# 2. Root directory of the Ye et al. dataset
YE_ET_AL_directory = '/home/cs21d002_eashaan/PhD/Objective1/Resources/Ye_et_al_Dataset'

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/cs21d002_eashaan/PhD/Objective1/data/repos'

# Creating full dataframe of the dataset
ye_et_al_df = process_ye_et_al_dataset(YE_ET_AL_directory, CLONE_DIR)

# Save the whole ye_et_al dataset
ye_et_al_df.to_csv('/home/cs21d002_eashaan/PhD/Objective1/benchmark_dataset_analysis/metadata/ye_et_al_full_data.csv', index=False)
print(f" Whole Ye et al. dataset collection and saved in a CSV file....")

# 4. Path for the final output CqSV file.
OUTPUT_CSV_PATH = '/home/cs21d002_eashaan/PhD/Objective1/benchmark_dataset_analysis/metadata/ye_metadata.csv'

# 5. Extensions for Polyglot Index Calculation
LANGUAGE_EXTENSIONS = {
    'c++': ['.c', '.cc', '.cmake', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx', '.in', '.json', '.make', '.py', '.sh', '.xml'],
    'go': ['.go', '.json', '.proto', '.sh', '.yaml', '.yml'],
    'java': ['.gradle', '.groovy', '.java', '.json', '.properties', '.xml', '.yml', '.yaml'],
    'javascript': ['.css', '.html', '.js', '.json', '.jsx', '.mjs', '.scss', '.sh', '.ts', '.tsx', '.yaml', '.yml'],
    'kotlin': ['.gradle', '.json', '.kt', '.kts', '.properties', '.xml', '.yaml', '.yml'],
    'python': ['.bash', '.cfg', '.in', '.ini', '.json', '.py', '.sh', '.toml', '.yaml', '.yml']
}
PRIMARY_EXTENSIONS = {
    'python': ['.py'],
    'java': ['.java'],
    'kotlin': ['.kt'],
    'c++': ['.c', '.cc', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx'],
    'go': ['.go'],
    'javascript': ['.js', '.jsx', '.mjs', '.ts', '.tsx']
}

Processing Ye et al. Dataset
Successfully loaded 22747 total bug reports.


Finding Parent Commits:  33%|███▎      | 2/6 [00:00<00:00, 16.03it/s]

An error occurred while processing apache/tomcat: Ref 'b022c57' did not resolve to an object
An error occurred while processing eclipse-aspectj/aspectj: Ref 'd916002' did not resolve to an object


Finding Parent Commits:  33%|███▎      | 2/6 [00:01<00:00, 16.03it/s]

An error occurred while processing eclipse-birt/birt: Ref '680c047' did not resolve to an object


Finding Parent Commits:  67%|██████▋   | 4/6 [00:02<00:01,  1.55it/s]

An error occurred while processing eclipse-jdt/eclipse.jdt.ui: Ref 'c86e147' did not resolve to an object
An error occurred while processing eclipse-platform/eclipse.platform.swt: Ref '1f7643c' did not resolve to an object


Finding Parent Commits: 100%|██████████| 6/6 [00:03<00:00,  1.93it/s]


An error occurred while processing eclipse-platform/eclipse.platform.ui: Ref '61bd659' did not resolve to an object

 Removed 14233 rows with missing 'before_fix_sha' values
Processing complete....
 Whole Ye et al. dataset collection and saved in a CSV file....


## Helper Methods

In [25]:
# Pre-requisite: Assume 'ye_et_al_df' is loaded and has the 'report_time' column
print(" Verifying the corrected 'report_time' column")

# The 'report_time' column should already be in datetime formate for your pre-processing
# we will just ensure it, coercing any potential errors that might still exist
ye_et_al_df['report_time'] = pd.to_datetime(ye_et_al_df['report_time'])

# 1. Show basic statistics of the corrected dates
print("Overall corrected date statistics:")
# Filter out any NaT values for accurate stats
valid_dates = ye_et_al_df['report_time'].dropna()
if not valid_dates.empty:
    print(f" Earliest Date: {valid_dates.min()}")
    print(f" Latest Date: {valid_dates.max()}")
    print(f" Median Date: {valid_dates.median()}")
else:
    print(" No valid dates found in the 'report_time' column.")

# 2. Show the distribution of bug reports by year
print("Distribution of Bug Reports per Year (top 15):")
print(valid_dates.dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 3. Isolate and check if any '1970' dates remain
print("Checking for any remaining anomalous '1970' dates")
problematic_rows = ye_et_al_df[ye_et_al_df['report_time'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Warning: Found {len(problematic_rows)} entries that still have a '1970' year")
else:
    print("Success: No entries with a '1970' year were found in the 'report_time' column.")

 Verifying the corrected 'report_time' column
Overall corrected date statistics:
 Earliest Date: 2001-10-10 22:35:04
 Latest Date: 2014-01-18 12:42:47
 Median Date: 2008-09-18 17:34:38
Distribution of Bug Reports per Year (top 15):
report_time
2014      20
2013     357
2012     365
2011     517
2010    1236
2009    1502
2008    1327
2007    1667
2006    1242
2005     167
2004      68
2003      26
2002      17
2001       3
Checking for any remaining anomalous '1970' dates
Success: No entries with a '1970' year were found in the 'report_time' column.


In [26]:
def count_lines_in_file(file_path):
    '''
    Counts lines in a file, handling encoding errors.
    '''
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return len(f.readlines())
    except Exception:
        return 0

In [27]:
def calculate_loc_and_polyglot(repo_path, declared_language):
    '''
    Calculates LoC and polyglot index. Auto-detects the actual primary language in the snapshot to handle
    migrations
    '''
    loc_by_lang = {lang: 0 for lang in PRIMARY_EXTENSIONS.keys()}
    loc_relevant = 0

    # first, calculate LoC for each potential primary language
    for root, _, files in os.walk(repo_path):
        for file in files:
            for lang, exts in PRIMARY_EXTENSIONS.items():
                if file.endswith(tuple(exts)):
                    loc_by_lang[lang] += count_lines_in_file(os.path.join(root, file))

    # Auto-detect the language with the most LoC in this snapshot
    actual_primary_languge = max(loc_by_lang, key=loc_by_lang.get) if loc_by_lang else declared_language
    loc_primary = loc_by_lang.get(actual_primary_languge, 0)

    # Now, calculate total relevant LoC based on the DECLARED ecosystem
    relevant_exts = tuple(LANGUAGE_EXTENSIONS.get(declared_language.lower(), []))

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(relevant_exts):
                loc_relevant += count_lines_in_file(os.path.join(root, file))
                
    polyglot_index = (loc_primary / loc_relevant) if loc_relevant > 0 else 0
    return loc_relevant, polyglot_index

In [28]:
def count_dependencies(repo_path, language):
    """
    Calculates a proxy for dependency complexity by summing the Lines of Code (LoC)
    of standard dependency files for the primary language.
    """
    loc_count = 0
    lang = language.lower()

    dependency_files = []
    if lang == 'python':
        dependency_files = ['requirements.txt', 'pyproject.toml']
    elif lang in ['java', 'kotlin']:
        dependency_files = ['pom.xml', 'build.gradle', 'build.gradle.kts']
    elif lang == 'c++':
        dependency_files = ['CMakeLists.txt', 'Makefile'] # Example for C++
    elif lang == 'javascript':
        dependency_files = ['package.json'] # Example for JS
    elif lang == 'go':
        dependency_files = ['go.mod'] # Example for Go

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file in dependency_files:
                file_path = os.path.join(root, file)
                # Simply add the number of lines in the file to the count
                loc_count += count_lines_in_file(file_path)
    
    return loc_count

In [29]:
def calculate_average_complexity(repo_path, language):
    '''
    Calculates average cyclomatic complexity using the 'lizard' tool.
    We store the ouput of lizard to a temporary file to handle large outputs reliably.
    '''
    # Create a temp file to stroe the lizard output
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt', encoding='utf-8') as temp_out:
        temp_filename = temp_out.name

    try:
        # lizard expects 'c++' to be written as 'cpp'
        lang_for_lizard = 'cpp' if language.lower() == 'c++' else language.lower()
        # print("Language for lizard: ", lang_for_lizard)

        # Redirect stdout to the temp file
        # We run the command and tell it to write its output directly to our temp file
        result = subprocess.run(
            ['lizard', '-i', '0', repo_path], # another command ['lizard', '-l', 'lang_for_lizard', repo_path]
            stdout = open(temp_filename, 'w',  encoding='utf-8'), # Write stdout to the temp file
            stderr=subprocess.PIPE, # Still capture any errors in memory 
            check=False, text=True
        )

        # We can still manually check the return code if we want to log detailed errors
        if result.returncode != 0:
            print(f"\nWarning: Lizard finished with a non-zero exit code ({result.returncode}) for {repo_path}. This usually indicates warnings were found. Continuing to parse output.")
            # We don't return here, because the output file is likely still valid.

        # Read the results back from the file
        with open(temp_filename, 'r', encoding='utf-8') as f:
            lizard_output = f.read()

        # Get all non-empty lines from the output
        lines = [line for line in lizard_output.strip().splitlines()]

        # the summary data is on the second to last line
        if len(lines) >=3 :
            # target the line with the numbers (the last non-empty line)
            summary_line = lines[-1]

            # Split the line by whitespace
            values = summary_line.split()

            if len(values) >= 3:
                # The Avg CCN is the 3rd value (index 2)
                avg_ccn = float(values[2])
                return avg_ccn

        # Find the summary line in the output
        # If we reach here, the summary line was not found or was malformed
        print(f"\nWarning: Could not parse lizard summary for {repo_path}.")
        return 0.0
        
    except FileNotFoundError:
        # This error is critical, so we print it once and then it will return 0 for others.
        print("\nERROR: 'lizard' command not found. Please install it with 'pip install lizard'.")
        return 0.0
    except subprocess.CalledProcessError as e:
        print(f"\n Lizard command failed for {repo_path}. Stderr: {e.stderr}")
        return 0.0
    except (IndexError, ValueError) as e:
        print(f"\nFailed to extract complexity value from summary line for {repo_path}. Error: {e}")
        return 0.0
    except Exception as e:
        print(f"\nAn unexpected error occurred in calculate_average_complexity: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

## Main Code

In [31]:
print("Starting metadata extraction for Ye et al. Dataset...")

# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

results = []
ye_et_al_df['bug_report'] = ye_et_al_df['summary'] + "\n" + ye_et_al_df['description']

for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos"):
    repo_name = row['repo_name']
    language = row['language']
    unique_bugs = row['total_unique_bug_report']

    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    print(f"Calculating meta data for repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    repo_ye_data = ye_et_al_df[ye_et_al_df['repo_name'] == repo_name].copy()
    if repo_ye_data.empty:
        print(f"Warning: No data found for {repo_name} in YE et al main dataframe. Skipping.")
        continue

    # Get the snapshot commit from the latest bug report
    latest_bug = repo_ye_data.sort_values(by='report_time', ascending=False).iloc[0]
    snapshot_commit = latest_bug['before_fix_sha']

    try:
        repo = git.Repo(repo_path)
        repo.git.checkout(snapshot_commit, f=True)

        # *****************
        # Calculate Metrics
        # *****************

        # c) Age
        # --- Metrics now use the reliable 'report_time' column ---
        min_date = repo_ye_data['report_time'].min()
        max_date = repo_ye_data['report_time'].max()
        age_years = ((max_date - min_date).days) / 365.25
        median_bug_year = repo_ye_data['report_time'].dt.year.median()

        # d) No. of authors & e) No. of commits (Correctly scoped to the snapshot)
        all_commits = list(repo.iter_commits())
        num_commits = len(all_commits)
        num_authors = len({c.author.email for c in all_commits})

        # a) LoC & g) Polyglot Index
        loc, polyglot_index = calculate_loc_and_polyglot(repo_path, language)

        # f) No. of external dependencies
        dependencies = count_dependencies(repo_path, language)

        # h) Bug density
        kloc = loc / 1000
        bug_density = (kloc / unique_bugs) if unique_bugs > 0 else 0

        # i) code_complexity
        code_complexity = calculate_average_complexity(repo_path, language)

        # j) Calculate bug report verbosoty
        bug_report_verbosity = repo_ye_data['bug_report'].str.split().str.len().mean()
        
        results.append({
            'repo_name': repo_name,
            'language': language,
            'LoC': loc,
            'age_years': age_years,
            'median_bug_year': median_bug_year, # Raw data for later categorization
            'num_authors': num_authors,
            'num_commits': num_commits,
            'num_dependencies': dependencies,
            'polyglot_index': polyglot_index,
            'bug_density': bug_density,
            'code_complexity': code_complexity,
            'bug_report_verbosity': bug_report_verbosity
        })
    
    # except git.exec.GitCommandError as e:
    #     print(f"Error processing Git Repo {repo_name}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {repo_name}: {e}")

if not results:
    print("No results were generated.")
    exit(0)

# Create final DataFrame
final_df = pd.DataFrame(results)

# # b) Calculate project_size category
# final_df = final_df.groupby('language', group_keys=False).apply(categorize_by_tercile)

# Save to CSV
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Metdata extraction complete. Results saved to '{OUTPUT_CSV_PATH}'")


Starting metadata extraction for Ye et al. Dataset...


Processing Repos:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating meta data for repo: apache/tomcat


Processing Repos:  17%|█▋        | 1/6 [00:11<00:55, 11.16s/it]


Calculating meta data for repo: eclipse-aspectj/aspectj


...


Calculating meta data for repo: eclipse-platform/eclipse.platform.swt


Processing Repos:  83%|████████▎ | 5/6 [02:22<00:30, 30.54s/it]


Calculating meta data for repo: eclipse-platform/eclipse.platform.ui


Processing Repos: 100%|██████████| 6/6 [02:48<00:00, 28.13s/it]


Metdata extraction complete. Results saved to '/home/cs21d002_eashaan/PhD/Objective1/benchmark_dataset_analysis/metadata/ye_metadata.csv'


In [15]:
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import glob
import tempfile 

In [16]:
def process_ye_et_al_dataset(data_directory, clone_dir):
    '''
    Loads, combines, and processes the Ye et al. dataset to prepare it for metadata extraction.
    This method will:
    a) Load data from all .xlsx files.
    b) Map folder names to correct 'owner/repo' names
    c) Add a 'language' column.
    d) Rename the 'commit' column to 'after_fix_sha'
    e) Find the parent commit for each fix to create the 'before_fix_sha'

    Args:
        data_directory (str): The path to the folder containing the .xlsx files.
        clone_dir (str): The path to the parent directory where repos are cloned.

    Returns:
        pd.DataFrame: A fully processed DataFrame ready for analysis/
    '''
    print("Processing Ye et al. Dataset")

    # The manual mapping from filename to correct repo name
    repo_name_map = {
        'tomcat': 'apache/tomcat',
        'aspectj': 'eclipse-aspectj/aspectj',
        'birt': 'eclipse-birt/birt',
        'jdt': 'eclipse-jdt/eclipse.jdt.ui',
        'eclipse_platform_ui': 'eclipse-platform/eclipse.platform.ui',
        'swt': 'eclipse-platform/eclipse.platform.swt'
    }

    # 1. Load and combine data from all .xlsx files
    excel_files = glob.glob(os.path.join(data_directory, '*.xlsx'))
    if not excel_files:
        print(f"Error: No .xlsx files found in '{data_directory}'.")
        return None
    
    all_project_dfs = []
    for file_path in excel_files:
        project_key = os.path.basename(file_path).split('.')[0]
        df = pd.read_excel(file_path)
        df['project_key'] = project_key # Temporary key for mapping
        all_project_dfs.append(df)

    ye_et_al_df = pd.concat(all_project_dfs, ignore_index=True)
    print(f"Successfully loaded {len(ye_et_al_df)} total bug reports.")

    # 2. Add and rename columns
    ye_et_al_df['language'] = 'java'
    ye_et_al_df['repo_name'] = ye_et_al_df['project_key'].map(repo_name_map)
    ye_et_al_df.rename(columns={'commit': 'after_fix_sha'}, inplace=True)
    ye_et_al_df['bug_id'] = ye_et_al_df['project_key'].astype(str) + '-' + ye_et_al_df['bug_id'].astype(str)
    ye_et_al_df.drop(columns=['project_key'], inplace=True) # Clean up temp key

    # 3. Find the 'before_fix_sha' for each bug report
    ye_et_al_df['before_fix_sha'] = None
    grouped = ye_et_al_df.groupby('repo_name')

    for repo_name, group in tqdm(grouped, desc='Finding Parent Commits'):
        repo_path = os.path.join(clone_dir, 'java', repo_name.replace('/', '_'))
        
        if not os.path.exists(repo_path):
            tqdm.write(f"Warning: Cloned repo not found for {repo_name}. Skipping...")
            continue

        try:
            repo = git.Repo(repo_path)
            for index, row in group.iterrows():
                fix_commit_sha = row['after_fix_sha']
                try:
                    # Get the commit object for the fix
                    commit = repo.commit(fix_commit_sha)
                    # The first parent is the before_fix_sha
                    if commit.parents:
                        parent_sha = commit.parents[0].hexsha
                        ye_et_al_df.loc[index, 'before_fix_sha'] = parent_sha
                except git.exc.GitCommandError:
                    # This happens if the commit hash doesn't exist in the repo
                    tqdm.write(f"Warning: Commit {fix_commit_sha} not found in {repo_name}.")
                    continue
        
        except Exception as e:
            tqdm.write(f"An error occurred while processing {repo_name}: {e}")

    print("Processing complete....")
    return ye_et_al_df

In [17]:
# 2. Root directory of the Ye et al. dataset
YE_ET_AL_directory = '/home/cs21d002_eashaan/PhD/Objective1/Resources/Ye_et_al_Dataset'

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/cs21d002_eashaan/PhD/Objective1/data/repos'

# Creating full dataframe of the dataset
ye_et_al_df = process_ye_et_al_dataset(YE_ET_AL_directory, CLONE_DIR)

Processing Ye et al. Dataset
Successfully loaded 22747 total bug reports.


Finding Parent Commits:  17%|█▋        | 1/6 [00:00<00:01,  3.90it/s]

An error occurred while processing apache/tomcat: Ref 'b022c57' did not resolve to an object


Finding Parent Commits:  33%|███▎      | 2/6 [00:00<00:01,  3.78it/s]

An error occurred while processing eclipse-aspectj/aspectj: Ref 'd916002' did not resolve to an object


...

An error occurred while processing eclipse-birt/birt: Ref '680c047' did not resolve to an object


Finding Parent Commits:  83%|████████▎ | 5/6 [00:03<00:00,  1.45it/s]

An error occurred while processing eclipse-jdt/eclipse.jdt.ui: Ref 'c86e147' did not resolve to an object
An error occurred while processing eclipse-platform/eclipse.platform.swt: Ref '1f7643c' did not resolve to an object


Finding Parent Commits: 100%|██████████| 6/6 [00:04<00:00,  1.31it/s]

An error occurred while processing eclipse-platform/eclipse.platform.ui: Ref '61bd659' did not resolve to an object
Processing complete....


In [18]:
def check_missing_shas(df):
    """
    Checks and reports the number of rows missing a 'before_fix_sha'.
    """
    print("\n--- Checking for Missing 'before_fix_sha' Values ---")
    
    # Count how many values in the column are null or None
    missing_count = df['before_fix_sha'].isnull().sum()
    total_count = len(df)
    
    if total_count > 0:
        missing_percentage = (missing_count / total_count) * 100
        print(f"Total Bug Reports: {total_count}")
        print(f"Number of Missing SHAs: {missing_count}")
        print(f"Percentage Missing: {missing_percentage:.2f}%")
    else:
        print("The DataFrame is empty.")


In [19]:
# Run the check
check_missing_shas(ye_et_al_df)


--- Checking for Missing 'before_fix_sha' Values ---
Total Bug Reports: 22747
Number of Missing SHAs: 14233
Percentage Missing: 62.57%


In [20]:
def analyze_remaining_bugs(df):
    """
    Analyzes and reports how many bug reports remain for each project
    after removing rows with missing 'before_fix_sha'.
    """
    print("\n--- Bug Report Counts After Removing Missing SHAs ---")

    # Group by repository and count the total number of bug reports
    total_counts = df.groupby('repo_name')['bug_id'].count()

    # Drop rows where 'before_fix_sha' is null, then group and count again
    remaining_counts = df.dropna(subset=['before_fix_sha']).groupby('repo_name')['bug_id'].count()

    # Combine the counts into a new DataFrame for easy comparison
    summary_df = pd.DataFrame({
        'Total_Bugs_Originally': total_counts,
        'Bugs_Remaining': remaining_counts
    }).fillna(0) # Fill projects with 0 remaining bugs

    # Calculate the percentage of data lost
    summary_df['Bugs_Lost'] = summary_df['Total_Bugs_Originally'] - summary_df['Bugs_Remaining']
    summary_df['Percentage_Lost'] = (summary_df['Bugs_Lost'] / summary_df['Total_Bugs_Originally']) * 100
    
    # Convert counts to integers for clean display
    summary_df['Bugs_Remaining'] = summary_df['Bugs_Remaining'].astype(int)

    # Sort by the number of remaining bugs
    sorted_summary = summary_df.sort_values(by='Bugs_Remaining', ascending=False)
    
    print(sorted_summary.to_string())

In [21]:
analyze_remaining_bugs(ye_et_al_df)


--- Bug Report Counts After Removing Missing SHAs ---
                                       Total_Bugs_Originally  Bugs_Remaining  Bugs_Lost  Percentage_Lost
repo_name                                                                                               
eclipse-birt/birt                                       4178            3498        680        16.275730
eclipse-jdt/eclipse.jdt.ui                              6274            2543       3731        59.467644
eclipse-platform/eclipse.platform.ui                    6495            2063       4432        68.237105
eclipse-aspectj/aspectj                                  593             269        324        54.637437
eclipse-platform/eclipse.platform.swt                   4151              90       4061        97.831848
apache/tomcat                                           1056              51       1005        95.170455
